# Undiagnosed Hypertension and Diabetes Screening with NHANES
## Part 3: Decision curves, fairness and SHAP stability

Upload `nhanes_analytic.csv` (from Part 1) when the first cell asks. Runtime is about 20 to 40 minutes,
mostly the 80 SHAP refits per outcome in Step 3.5.

### Step 3.1: Setup, data and shared functions (same as Part 2)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "lightgbm", "interpret", "shap"], check=False)
import os, re, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from interpret.glassbox import ExplainableBoostingClassifier

warnings.filterwarnings("ignore")
SEED = 2026
np.random.seed(SEED)
OUT = "/content/part3_out" if os.path.isdir("/content") else "./part3_out"
os.makedirs(OUT, exist_ok=True)

CSV = "/content/nhanes_analytic.csv" if os.path.isdir("/content") else "nhanes_analytic.csv"
if not os.path.exists(CSV):
    from google.colab import files
    print("Upload nhanes_analytic.csv from Part 1")
    up = files.upload()
    CSV = "/content/" + list(up.keys())[0]
data = pd.read_csv(CSV)
print(data.shape)

### Step 3.1.2: Predictor sets
- **Diabetes, main model:** usable in every cycle, so it can be tested on 2021-2023
- **Diabetes, extended model:** adds family history and gestational diabetes (not asked in 2021-2023), evaluated on 2017-2020 only
- **Hypertension:** blood pressure is excluded because it defines the outcome

In [ ]:
data["hx_htn"] = data["told_htn"]                      # ever told high blood pressure (1/0)
data["hx_dm"] = (data["told_dm"] == 1).astype(float)   # ever told diabetes
data.loc[data["told_dm"].isna(), "hx_dm"] = np.nan
data["smoking"] = data["smoking"].fillna("missing")

NUM_DM = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sbp_mean", "dbp_mean", "sedentary_min"]
BIN_DM = ["sex_female", "hx_htn", "cvd_history", "active_leisure"]
CAT_DM = ["smoking", "education"]
NUM_HTN = ["age", "bmi", "waist_cm", "height_cm", "whtr", "sedentary_min"]
BIN_HTN = ["sex_female", "hx_dm", "cvd_history", "active_leisure"]
CAT_HTN = ["smoking", "education"]
EXTRA_DM = ["famhx_dm", "gdm"]

TASKS = {
    "diabetes": {"pop": "dm_at_risk_pop", "y": "undx_dm", "num": NUM_DM, "bin": BIN_DM, "cat": CAT_DM,
                 "simple": ["age", "sex_female", "bmi"]},
    "hypertension": {"pop": "htn_at_risk_pop", "y": "undx_htn", "num": NUM_HTN, "bin": BIN_HTN, "cat": CAT_HTN,
                     "simple": ["age", "sex_female", "bmi"]},
}

def task_data(task):
    t = TASKS[task]
    d = data[(data[t["pop"]] == 1) & data[t["y"]].notna()].copy()
    d["education"] = d["education"].map(lambda v: "missing" if pd.isna(v) else str(int(v)))
    return d

for task in TASKS:
    d = task_data(task)
    print(task, d.groupby("split")[TASKS[task]["y"]].agg(["size", "sum", "mean"]).round(4).to_dict())

### Step 3.1.3: Models
Tree models handle missing values natively; logistic regression and EBM use median/mode imputation
fitted on the training data only. No resampling is used, because oversampling distorts calibration.

In [ ]:
def preprocessor(num, bin_, cat, scale=True):
    num_pipe = [("imp", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_pipe.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_pipe), num),
        ("bin", SimpleImputer(strategy="most_frequent"), bin_),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", drop="first"))]), cat),
    ])

def tree_matrix(d, num, bin_, cat):
    X = d[num + bin_].copy()
    for c in cat:
        X = pd.concat([X, pd.get_dummies(d[c], prefix=c, dtype=float)], axis=1)
    # XGBoost rejects [, ] and < in column names, so keep names to letters, digits and underscores
    X.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", str(c)) for c in X.columns]
    return X

def make_model(name, params, num, bin_, cat):
    if name == "LR":
        return Pipeline([("pre", preprocessor(num, bin_, cat)), ("m", LogisticRegression(max_iter=2000, C=params.get("C", 1.0)))])
    if name == "EBM":
        return Pipeline([("pre", preprocessor(num, bin_, cat, scale=False)),
                         ("m", ExplainableBoostingClassifier(random_state=SEED, interactions=params.get("interactions", 10)))])
    if name == "XGBoost":
        return XGBClassifier(n_estimators=params["n"], max_depth=params["depth"], learning_rate=params["lr"],
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss",
                             random_state=SEED, n_jobs=-1)
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=params["n"], num_leaves=params["leaves"], learning_rate=params["lr"],
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, min_child_samples=30,
                              random_state=SEED, verbose=-1)

GRIDS = {
    "LR": [{"C": c} for c in [0.01, 0.1, 1.0]],
    "EBM": [{"interactions": i} for i in [0, 10]],
    "XGBoost": [{"n": n, "depth": dp, "lr": lr} for n in [200, 500] for dp in [2, 3, 4] for lr in [0.03]],
    "LightGBM": [{"n": n, "leaves": lv, "lr": lr} for n in [200, 500] for lv in [7, 15] for lr in [0.03]],
}
TREE = {"XGBoost", "LightGBM"}

def fit_predict(name, params, tr, te, num, bin_, cat, y):
    m = make_model(name, params, num, bin_, cat)
    if name in TREE:
        Xtr = tree_matrix(tr, num, bin_, cat)
        Xte = tree_matrix(te, num, bin_, cat).reindex(columns=Xtr.columns, fill_value=0)
        m.fit(Xtr, tr[y])
        return m, m.predict_proba(Xte)[:, 1]
    cols = num + bin_ + cat
    m.fit(tr[cols], tr[y])
    return m, m.predict_proba(te[cols])[:, 1]

### Step 3.1.4: Metrics

In [ ]:
def calib_int_slope(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp, y).coef_[0][0]
    # calibration-in-the-large: intercept with the linear predictor as offset
    import statsmodels.api as sm
    inter = sm.GLM(y, np.ones_like(lp), family=sm.families.Binomial(), offset=lp.ravel()).fit().params[0]
    return inter, slope

def boot_auc(y, p, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

def threshold_for_sensitivity(y, p, target=0.80):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr >= target)]

def evaluate(y, p, thr):
    y = np.asarray(y)
    lo, hi = boot_auc(y, p)
    inter, slope = calib_int_slope(y, p)
    flag = p >= thr
    tp, fp = (flag & (y == 1)).sum(), (flag & (y == 0)).sum()
    fn, tn = (~flag & (y == 1)).sum(), (~flag & (y == 0)).sum()
    return {"AUC": roc_auc_score(y, p), "AUC_lo": lo, "AUC_hi": hi, "AUPRC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p), "Cal_intercept": inter, "Cal_slope": slope,
            "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp), "PPV": tp / max(tp + fp, 1),
            "Flagged_%": 100 * flag.mean()}

### Step 3.2: Refit the final models on 2011-2020 and predict 2021-2023
Settings are the ones Part 2 chose on the validation cycle.

In [ ]:
BEST = {
    ("diabetes", "LR"): {"C": 1.0}, ("diabetes", "EBM"): {"interactions": 10},
    ("diabetes", "XGBoost"): {"n": 200, "depth": 2, "lr": 0.03}, ("diabetes", "LightGBM"): {"n": 200, "leaves": 7, "lr": 0.03},
    ("hypertension", "LR"): {"C": 1.0}, ("hypertension", "EBM"): {"interactions": 10},
    ("hypertension", "XGBoost"): {"n": 200, "depth": 3, "lr": 0.03}, ("hypertension", "LightGBM"): {"n": 200, "leaves": 7, "lr": 0.03},
}
FINAL, THRESH, PRED = {}, {}, {}
for task, t in TASKS.items():
    d = task_data(task)
    tr, va, te = d[d.split == "train"], d[d.split == "validation"], d[d.split == "test"]
    trva = pd.concat([tr, va])
    y = t["y"]
    _, pv = fit_predict("LR", {"C": 1.0}, tr, va, ["age", "bmi"], ["sex_female"], [], y)
    THRESH[(task, "Baseline LR")] = threshold_for_sensitivity(va[y], pv)
    _, PRED[(task, "Baseline LR")] = fit_predict("LR", {"C": 1.0}, trva, te, ["age", "bmi"], ["sex_female"], [], y)
    for name in ["LR", "EBM", "XGBoost", "LightGBM"]:
        p = BEST[(task, name)]
        _, pv = fit_predict(name, p, tr, va, t["num"], t["bin"], t["cat"], y)
        THRESH[(task, name)] = threshold_for_sensitivity(va[y], pv)
        FINAL[(task, name)], PRED[(task, name)] = fit_predict(name, p, trva, te, t["num"], t["bin"], t["cat"], y)
        print(f"{task:12s} {name:9s} test AUC {roc_auc_score(te[y], PRED[(task, name)]):.3f}")
MODELS = ["Baseline LR", "LR", "EBM", "XGBoost", "LightGBM"]

### Step 3.3: Decision curve analysis (Vickers and Elkin, 2006)
Net benefit = TP/n - FP/n x (pt / (1 - pt)), compared with testing everyone and testing no one.

In [ ]:
def net_benefit(y, p, pt):
    y = np.asarray(y); n = len(y); flag = p >= pt
    tp = (flag & (y == 1)).sum(); fp = (flag & (y == 0)).sum()
    return tp / n - fp / n * pt / (1 - pt)

DCA_RANGE = {"diabetes": np.arange(0.005, 0.1501, 0.0025), "hypertension": np.arange(0.01, 0.4001, 0.005)}
KEY_PT = {"diabetes": [0.02, 0.03, 0.05, 0.10], "hypertension": [0.05, 0.10, 0.15, 0.20]}
dca_rows, key_rows = [], []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, task in zip(axes, TASKS):
    te = task_data(task); te = te[te.split == "test"]; y = te[TASKS[task]["y"]].values
    prev = y.mean()
    for pt in DCA_RANGE[task]:
        row = {"task": task, "threshold": round(pt, 4), "Test everyone": prev - (1 - prev) * pt / (1 - pt), "Test no one": 0.0}
        for m in MODELS:
            row[m] = net_benefit(y, PRED[(task, m)], pt)
        dca_rows.append(row)
    dd = pd.DataFrame([r for r in dca_rows if r["task"] == task])
    for m in MODELS + ["Test everyone"]:
        ax.plot(dd["threshold"], dd[m], label=m, lw=1.6 if m in ("XGBoost", "LR") else 1.1,
                ls="--" if m == "Test everyone" else "-")
    ax.axhline(0, color="k", lw=0.8, label="Test no one")
    ax.set_ylim(-0.005 if task == "diabetes" else -0.02, prev * 1.1)
    ax.set(title=f"Decision curves, undiagnosed {task} (2021-2023)", xlabel="Threshold probability", ylabel="Net benefit")
    ax.legend(fontsize=8)
    for pt in KEY_PT[task]:
        r = {"task": task, "threshold": pt, "Test everyone": prev - (1 - prev) * pt / (1 - pt)}
        for m in MODELS:
            r[m] = net_benefit(y, PRED[(task, m)], pt)
        key_rows.append(r)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_decision_curves.png"), dpi=300); plt.show()
dca_key = pd.DataFrame(key_rows).round(4)
pd.DataFrame(dca_rows).round(5).to_csv(os.path.join(OUT, "decision_curves_full.csv"), index=False)
dca_key.to_csv(os.path.join(OUT, "decision_curves_key_thresholds.csv"), index=False)
print(dca_key.to_string(index=False))

### Step 3.4: Fairness, performance by subgroup on 2021-2023
For each subgroup: AUC with bootstrap CI, observed/expected ratio (calibration-in-the-large; 1 = perfect),
and sensitivity at the threshold chosen on the validation cycle.

In [ ]:
RACE = {1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White", 4: "Non-Hispanic Black", 6: "Non-Hispanic Asian", 7: "Other or multiracial"}
def subgroups(d):
    g = {}
    g["Sex: male"] = d["sex_female"] == 0
    g["Sex: female"] = d["sex_female"] == 1
    g["Age 20-39"] = d["age"] < 40
    g["Age 40-59"] = (d["age"] >= 40) & (d["age"] < 60)
    g["Age 60+"] = d["age"] >= 60
    for k, v in RACE.items():
        g[f"Race: {v}"] = d["race_eth"] == k
    g["Income: below 1.3 x poverty"] = d["pir"] < 1.3
    g["Income: 1.3 to 3.5"] = (d["pir"] >= 1.3) & (d["pir"] < 3.5)
    g["Income: 3.5 or more"] = d["pir"] >= 3.5
    return g

fair_rows = []
for task in TASKS:
    te = task_data(task); te = te[te.split == "test"]; y = te[TASKS[task]["y"]].values
    for m in ["LR", "XGBoost", "EBM"]:
        p = PRED[(task, m)]; thr = THRESH[(task, m)]
        for gname, mask in subgroups(te).items():
            mk = mask.values
            yy, pp = y[mk], p[mk]
            row = {"task": task, "model": m, "subgroup": gname, "n": int(mk.sum()), "cases": int(yy.sum())}
            if yy.sum() >= 10 and (len(yy) - yy.sum()) >= 10:
                lo, hi = boot_auc(yy, pp, n=500)
                row.update({"AUC": roc_auc_score(yy, pp), "AUC_lo": lo, "AUC_hi": hi})
            row["O/E ratio"] = yy.sum() / pp.sum()
            row["Sensitivity"] = ((pp >= thr) & (yy == 1)).sum() / max(yy.sum(), 1)
            row["Flagged_%"] = 100 * (pp >= thr).mean()
            fair_rows.append(row)
fair = pd.DataFrame(fair_rows).round(3)
fair.to_csv(os.path.join(OUT, "fairness_subgroups.csv"), index=False)
print("Subgroups with fewer than 10 cases or 10 non-cases get no AUC (too unstable).")
print(fair[fair.model == "XGBoost"].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharey=True)
for ax, task in zip(axes, TASKS):
    f = fair[(fair.task == task) & (fair.model == "XGBoost") & fair["AUC"].notna()].iloc[::-1]
    ax.errorbar(f["AUC"], range(len(f)), xerr=[f["AUC"] - f["AUC_lo"], f["AUC_hi"] - f["AUC"]], fmt="o", capsize=3)
    overall = roc_auc_score(*(lambda d: (d[TASKS[task]["y"]], PRED[(task, "XGBoost")]))(task_data(task)[lambda x: x.split == "test"]))
    ax.axvline(overall, ls="--", color="grey", lw=1)
    ax.set_yticks(range(len(f))); ax.set_yticklabels(f["subgroup"])
    ax.set(title=f"XGBoost AUC by subgroup, undiagnosed {task}", xlabel="AUC (95% CI); dashed line = overall")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_fairness_auc.png"), dpi=300); plt.show()

### Step 3.5: SHAP stability
Mean |SHAP| on the fixed 2021-2023 test set, with one-hot columns summed back to their original variable.
Stability is measured four ways:
1. **Bootstrap:** 50 resamples of the 2011-2020 training data, model refit each time (Riley and Collins approach)
2. **Random seeds:** 30 refits on the same data with different seeds
3. **Survey period:** model trained on 2011-2016 only versus 2017-2020 only
4. **Model type:** XGBoost SHAP versus LightGBM SHAP versus EBM term importance versus logistic regression |standardised coefficient|
Agreement is summarised with Kendall's W (0 = no agreement, 1 = identical rankings) and Spearman correlation.

In [ ]:
import shap
from scipy.stats import spearmanr, rankdata

def base_var(col, task):
    for c in TASKS[task]["cat"]:
        if col.startswith(c + "_"):
            return c
    return col

def xgb_importance(task, train_df, params, seed=SEED, X_eval=None):
    t = TASKS[task]
    m = XGBClassifier(n_estimators=params["n"], max_depth=params["depth"], learning_rate=params["lr"], subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=5, eval_metric="logloss", random_state=seed, n_jobs=-1)
    Xtr = tree_matrix(train_df, t["num"], t["bin"], t["cat"])
    m.fit(Xtr, train_df[t["y"]])
    Xe = X_eval.reindex(columns=Xtr.columns, fill_value=0)
    sv = shap.TreeExplainer(m).shap_values(Xe)
    imp = pd.Series(np.abs(sv).mean(axis=0), index=Xtr.columns)
    return imp.groupby([base_var(c, task) for c in imp.index]).sum()

def kendalls_w(rank_matrix):
    R = np.asarray(rank_matrix)  # rows = runs, cols = features
    m, n = R.shape
    S = ((R.sum(axis=0) - m * (n + 1) / 2) ** 2).sum()
    return 12 * S / (m ** 2 * (n ** 3 - n))

N_BOOT, N_SEED = 50, 30
stab_summary, rank_tables = [], {}
for task, t in TASKS.items():
    d = task_data(task)
    trva = d[d.split.isin(["train", "validation"])]
    te = d[d.split == "test"]
    Xte = tree_matrix(te, t["num"], t["bin"], t["cat"])
    params = BEST[(task, "XGBoost")]
    ref = xgb_importance(task, trva, params, X_eval=Xte)
    feats = ref.sort_values(ascending=False).index.tolist()
    rng = np.random.default_rng(SEED)

    boot = []
    for b in range(N_BOOT):
        idx = rng.integers(0, len(trva), len(trva))
        boot.append(xgb_importance(task, trva.iloc[idx], params, seed=SEED + b, X_eval=Xte).reindex(feats).fillna(0))
    B = pd.DataFrame(boot)
    seeds = pd.DataFrame([xgb_importance(task, trva, params, seed=s, X_eval=Xte).reindex(feats).fillna(0) for s in range(1, N_SEED + 1)])

    rank_b = B.apply(lambda r: rankdata(-r), axis=1, result_type="expand"); rank_b.columns = feats
    rank_s = seeds.apply(lambda r: rankdata(-r), axis=1, result_type="expand"); rank_s.columns = feats
    top5_ref = set(feats[:5])
    tab = pd.DataFrame({
        "mean_abs_SHAP": ref.reindex(feats).values,
        "rank_full_model": range(1, len(feats) + 1),
        "boot_median_rank": rank_b.median().values,
        "boot_rank_2.5%": rank_b.quantile(0.025).values,
        "boot_rank_97.5%": rank_b.quantile(0.975).values,
        "boot_%_in_top5": (rank_b <= 5).mean().values * 100,
        "seed_rank_range": (rank_s.max() - rank_s.min()).values,
    }, index=feats).round(3)
    rank_tables[task] = tab

    early = d[d.split == "train"]; late = d[d.split == "validation"]
    imp_early = xgb_importance(task, early, params, X_eval=Xte).reindex(feats).fillna(0)
    imp_late = xgb_importance(task, late, params, X_eval=Xte).reindex(feats).fillna(0)

    # model-type agreement
    lgbm = FINAL[(task, "LightGBM")]
    Xtr_cols = tree_matrix(trva, t["num"], t["bin"], t["cat"]).columns
    sv_l = shap.TreeExplainer(lgbm).shap_values(Xte.reindex(columns=Xtr_cols, fill_value=0))
    sv_l = sv_l[1] if isinstance(sv_l, list) else sv_l
    imp_lgbm = pd.Series(np.abs(sv_l).mean(axis=0), index=Xtr_cols)
    imp_lgbm = imp_lgbm.groupby([base_var(c, task) for c in imp_lgbm.index]).sum().reindex(feats).fillna(0)
    lr = FINAL[(task, "LR")]
    names = lr.named_steps["pre"].get_feature_names_out()
    coef = pd.Series(np.abs(lr.named_steps["m"].coef_[0]), index=names)
    def lr_var(n):
        n = n.split("__", 1)[1]
        n = n.replace("missingindicator_", "")
        for c in t["cat"]:
            if n.startswith(c + "_"): return c
        return n
    imp_lr = coef.groupby([lr_var(n) for n in coef.index]).sum().reindex(feats).fillna(0)
    ebm = FINAL[(task, "EBM")].named_steps["m"]
    ebm_names = FINAL[(task, "EBM")].named_steps["pre"].get_feature_names_out()
    ebm_imp = pd.Series(ebm.term_importances(), index=[" & ".join(ebm_names[i] for i in f) for f in ebm.term_features_])
    main = pd.Series([ebm_imp.get(n, 0.0) for n in ebm_names], index=[lr_var(n) for n in ebm_names])
    imp_ebm = main.groupby(level=0).sum().reindex(feats).fillna(0)  # sum one-hot levels and missing flags into one variable

    agree = pd.DataFrame({"XGBoost": ref.reindex(feats), "LightGBM": imp_lgbm, "EBM": imp_ebm, "LR": imp_lr})
    corr = agree.corr(method="spearman").round(3)
    W_models = kendalls_w(np.vstack([rankdata(-agree[c]) for c in agree]))
    stab_summary.append({
        "task": task, "features": len(feats),
        "Kendall_W_bootstrap": kendalls_w(rank_b.values), "Kendall_W_seeds": kendalls_w(rank_s.values),
        "Spearman_2011-16_vs_2017-20": spearmanr(imp_early, imp_late).correlation,
        "Top5_overlap_2011-16_vs_2017-20": len(set(imp_early.sort_values(ascending=False).index[:5]) & set(imp_late.sort_values(ascending=False).index[:5])),
        "Kendall_W_across_4_model_types": W_models,
        "Top5_from_full_model": ", ".join(feats[:5]),
    })
    print(f"\n=== {task}: SHAP rank stability (XGBoost) ===")
    print(tab.to_string())
    print(f"\nSpearman correlation of importance rankings across model types ({task}):")
    print(corr.to_string())
    tab.to_csv(os.path.join(OUT, f"shap_stability_{task}.csv"))
    corr.to_csv(os.path.join(OUT, f"importance_agreement_models_{task}.csv"))

stab = pd.DataFrame(stab_summary).round(3)
stab.to_csv(os.path.join(OUT, "shap_stability_summary.csv"), index=False)
print("\n=== Stability summary ===")
print(stab.T.to_string())

### Step 3.6: Stability figure (bootstrap rank distribution of each feature)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, task in zip(axes, TASKS):
    tab = rank_tables[task]
    yv = range(len(tab))[::-1]
    ax.errorbar(tab["boot_median_rank"], list(yv), xerr=[tab["boot_median_rank"] - tab["boot_rank_2.5%"], tab["boot_rank_97.5%"] - tab["boot_median_rank"]],
                fmt="o", capsize=3)
    ax.set_yticks(list(yv)); ax.set_yticklabels(tab.index)
    ax.invert_xaxis(); ax.set(title=f"SHAP rank across {N_BOOT} bootstrap refits, {task}", xlabel="Rank (1 = most important), median and 95% range")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_shap_stability.png"), dpi=300); plt.show()

### Step 3.7: Save everything

In [ ]:
print("Saved to", OUT, os.listdir(OUT))
try:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/part3_results", "zip", OUT)
    files.download("/content/part3_results.zip")
except Exception:
    pass

### What to send back
The zip file `part3_results.zip`, or the printed tables from Steps 3.3, 3.4 and 3.5.